## Setup

This notebook requires Python 3.11 and the packages below. Run this command in a notebook cell once if they are not already installed:

```python
%pip install numpy==2.2.6 pandas==2.3.3
```

Restart the kernel after installation, then run the notebook from top to bottom.

In [1]:
#| label: setup
#| echo: true

import pandas as pd
from IPython.display import display

from example_data import (
    learn_shared_transformation,
    make_example_tables,
    transform_to_metric_view,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

## Learning goals

By the end of this introduction, you should be able to:

- distinguish an original table from a complete metric-ready representation;
- construct and interpret the missingness indicator matrices $M^X$ and $M^Y$;
- explain why one shared transformation must be used for $X$ and $Y$;
- explain how numerical and categorical values enter $\widetilde X$ or $\widetilde Y$; and
- identify which future metrics inspect the original tables and which use complete representations.

## 1. Two tables with one schema

Let

$$
X=(x_{ij}),\qquad i=1,\ldots,n_X,\quad j=1,\ldots,d
$$

and

$$
Y=(y_{ij}),\qquad i=1,\ldots,n_Y,\quad j=1,\ldots,d
$$

The tables share the same $d$ variables and the same datatype for each variable. Therefore, $X$ has dimensions $n_X\times d$, and $Y$ has dimensions $n_Y\times d$. They do **not** need the same number of records, and their values may differ. For variable $j$ with value space $\mathcal X_j$,

$$
x_{ij},y_{ij}\in\mathcal X_j\cup\{\mathrm{NA}\}
$$

The symbol $\mathrm{NA}$ denotes a missing value. The names $X$ and $Y$ are deliberately neutral: either table could be observed, synthetic, reconstructed, or produced by another process.

In [2]:
#| label: construct-small-tables

X, Y = make_example_tables()

print(f"X has {len(X)} rows and {X.shape[1]} columns")
display(X)
print(f"Y has {len(Y)} rows and {Y.shape[1]} columns")
display(Y)

X has 6 rows and 4 columns


,age,income_k,region,outcome
x1,24,32.000,North,0
x2,31,48.000,South,0
x3,<NA>,55.000,North,1
x4,46,<NA>,West,1
x5,58,91.000,<NA>,1
x6,39,67.000,South,0


Y has 5 rows and 4 columns


,age,income_k,region,outcome
y1,25,35.000,North,0
y2,<NA>,51.000,South,1
y3,44,<NA>,West,1
y4,60,88.000,<NA>,1
y5,37,64.000,East,0


The example is intentionally small enough to inspect by eye:

- `age` and `income_k` are numerical;
- `region` is categorical;
- `outcome` is binary and will later play the role of a prediction target;
- both tables have missing values, but not in the same cells; and
- `East` occurs only in $Y$, illustrating why category definitions must be shared.

In [3]:
#| label: verify-shared-schema

schema = pd.DataFrame(
    {
        "X dtype": X.dtypes.astype(str),
        "Y dtype": Y.dtypes.astype(str),
        "same dtype?": X.dtypes.astype(str).eq(Y.dtypes.astype(str)),
    }
)
display(schema)
assert X.columns.equals(Y.columns)
assert X.dtypes.astype(str).equals(Y.dtypes.astype(str))

,X dtype,Y dtype,same dtype?
age,Int64,Int64,True
income_k,Float64,Float64,True
region,string,string,True
outcome,Int64,Int64,True


## 2. Missingness indicator matrices $M^X$ and $M^Y$

The missingness masks record **where** values are absent without filling them. Define

$$
(M^X)_{ij}=\mathbf 1\{x_{ij}=\mathrm{NA}\}
=
\begin{cases}
1, & x_{ij}\text{ is missing},\\
0, & x_{ij}\text{ is observed},
\end{cases}
$$

Define $M^Y$ in exactly the same way. Thus, $M^X$ has dimensions $n_X\times d$, and $M^Y$ has dimensions $n_Y\times d$.

In [4]:
#| label: missingness-masks

M_X = X.isna().astype("int8")
M_Y = Y.isna().astype("int8")

print("M^X: 1 means the corresponding cell in X is missing")
display(M_X)
print("M^Y: 1 means the corresponding cell in Y is missing")
display(M_Y)

M^X: 1 means the corresponding cell in X is missing


,age,income_k,region,outcome
x1,0,0,0,0
x2,0,0,0,0
x3,1,0,0,0
x4,0,1,0,0
x5,0,0,1,0
x6,0,0,0,0


M^Y: 1 means the corresponding cell in Y is missing


,age,income_k,region,outcome
y1,0,0,0,0
y2,1,0,0,0
y3,0,1,0,0
y4,0,0,1,0
y5,0,0,0,0


A natively complete table has a mask containing only zeros. For incomplete tables, the masks let us compare missingness without confusing an absent value with a numerical zero or an ordinary category.

## 3. One shared transformation

Many value-, utility-, and privacy-based metrics need complete numerical inputs. We first pool $X$ and $Y$ **only to learn shared preprocessing rules**. We do not merge their rows when computing the metrics. We then apply the same learned rules separately to both tables:

$$
\widetilde X=T_{X\cup Y}(X),
\qquad
\widetilde Y=T_{X\cup Y}(Y)
$$

Let $\widetilde d$ be the number of columns in the complete preprocessed tables. Therefore, $\widetilde X$ has dimensions $n_X\times\widetilde d$, and $\widetilde Y$ has dimensions $n_Y\times\widetilde d$.

The matrices $M^X$ and $M^Y$ have already recorded the locations of the missing cells. For this introductory example, $T_{X\cup Y}$ then does the following:

1. learns the pooled median, minimum, and maximum of each numerical column;
2. replaces each missing numerical value with its pooled median;
3. replaces each missing categorical value with the special category label `__MISSING__`;
4. learns one shared category-to-code mapping from $X$ and $Y$;
5. scales the filled numerical values using their pooled minima and maxima; and
6. copies the relevant columns of $M^X$ and $M^Y$ into the complete representations so the locations of the original numerical missing values are retained.

Fitting unrelated transformations to $X$ and $Y$ could assign different scales or category codes to the same value, making the comparison meaningless.

In [5]:
#| label: fit-shared-transformation

numeric_parameters, region_to_code, region_mapping = (
    learn_shared_transformation(X, Y)
)

print("Numerical parameters learned from X and Y together")
display(numeric_parameters)
print("Shared category-to-code mapping learned from X and Y together")
display(region_mapping)

Numerical parameters learned from X and Y together


,median,minimum,maximum,range
age,39.000,24.000,60.000,36.000
income_k,55.000,32.000,91.000,59.000


Shared category-to-code mapping learned from X and Y together


,region level,shared code
0,East,0
1,North,1
2,South,2
3,West,3
4,__MISSING__,4


For each numerical cell, the transformation is easiest to read as two steps:

$$
\text{filled value}=
\begin{cases}
\text{original value}, & \text{if it is observed},\\
\text{pooled median}, & \text{if it is missing},
\end{cases}
$$

followed by

$$
\text{scaled value}
=\frac{\text{filled value}-\text{pooled minimum}}
{\text{pooled maximum}-\text{pooled minimum}}
$$

There is no need to introduce a second missingness symbol such as $m_{ij}$. The numerical missingness-indicator columns in $\widetilde X$ and $\widetilde Y$ are copied directly from the corresponding columns of $M^X$ and $M^Y$. For categorical columns, the special category label `__MISSING__` retains the missing locations.

In [6]:
#| label: apply-shared-transformation

X_tilde = transform_to_metric_view(X, numeric_parameters, region_to_code)
Y_tilde = transform_to_metric_view(Y, numeric_parameters, region_to_code)

print("X_tilde: the complete metric-ready representation of X")
display(X_tilde)
print("Y_tilde: the complete metric-ready representation of Y")
display(Y_tilde)

X_tilde: the complete metric-ready representation of X


,age,age__missing,income_k,income_k__missing,region__code,outcome
x1,0.000,0,0.000,0,1,0
x2,0.194,0,0.271,0,2,0
x3,0.417,1,0.390,0,1,1
x4,0.611,0,0.390,1,3,1
x5,0.944,0,1.000,0,4,1
x6,0.417,0,0.593,0,2,0


Y_tilde: the complete metric-ready representation of Y


,age,age__missing,income_k,income_k__missing,region__code,outcome
y1,0.028,0,0.051,0,1,0
y2,0.417,1,0.322,0,2,1
y3,0.556,0,0.390,1,3,1
y4,1.000,0,0.949,0,4,1
y5,0.361,0,0.542,0,0,0


In [7]:
#| label: verify-complete-representations

assert X_tilde.columns.equals(Y_tilde.columns)
assert not X_tilde.isna().any().any()
assert not Y_tilde.isna().any().any()

print("Same transformed columns:", X_tilde.columns.equals(Y_tilde.columns))
print("Missing cells in X_tilde:", int(X_tilde.isna().sum().sum()))
print("Missing cells in Y_tilde:", int(Y_tilde.isna().sum().sum()))

Same transformed columns: True
Missing cells in X_tilde: 0
Missing cells in Y_tilde: 0


> **Important:** $\widetilde X$ and $\widetilde Y$ are metric-ready views, not claims about the unknown true values. Median filling here makes a numerical representation complete; the indicator columns and explicit missing category retain information about where values were absent.

## 4. The two metric branches

The series will build individual metrics later. For now, the important split is the input view each family needs.

| Input view | Notation | What it preserves | Metric families introduced later |
|---|---|---|---|
| Original tables | $X,Y$ | Original values and actual missing cells | Missingness fidelity |
| Missingness indicator matrices | $M^X,M^Y$ | Locations and row-wise patterns of missing values | Marginal and joint missingness |
| Complete metric-ready tables | $\widetilde X,\widetilde Y$ | Aligned numerical values plus explicit missingness information | Data fidelity, utility, and privacy |

The relationship can be summarized as

$$
(X,Y)
\longrightarrow
\begin{cases}
(M^X,M^Y), & \text{missingness metrics},\\[4pt]
(\widetilde X,\widetilde Y)=
\bigl(T_{X\cup Y}(X),T_{X\cup Y}(Y)\bigr),
& \text{value, utility, and privacy metrics}.
\end{cases}
$$

The utility lessons use the real holdout $U$, with dimensions $n_U\times d$. The missingness-dependence and membership lessons use the real missingness holdout $V$, with dimensions $n_V\times d$. Their missingness indicator matrices are $M^U$ and $M^V$. The complete preprocessed holdouts are $\widetilde U=T_{X\cup Y}(U)$ and $\widetilde V=T_{X\cup Y}(V)$. They have dimensions $n_U\times\widetilde d$ and $n_V\times\widetilde d$. Neither holdout is used to fit the transformation.

### Complete versus incomplete original tables

- If $X$ and $Y$ are already complete, then $M^X$ and $M^Y$ contain only zeros. The complete-representation metrics remain applicable, while missingness metrics carry no additional information.
- If either table is incomplete, both branches matter: the original tables reveal the missingness process, while $\widetilde X$ and $\widetilde Y$ provide aligned complete inputs for the metrics that require them.

This is why we do not introduce separate symbols such as $X_c$ and $X_n$. One table can have both an original incomplete view and a complete metric-ready view.

### Reusing the same example in later notebooks

The definitions live in [`example_data.py`](example_data.py), not in another notebook. Every subsequent lesson can therefore create fresh copies of the exact same objects without executing this introductory notebook:

```python
from example_data import make_example_views

example = make_example_views()
X, Y = example.X, example.Y
U, V = example.U, example.V
M_X, M_Y = example.M_X, example.M_Y
M_U, M_V = example.M_U, example.M_V
X_tilde, Y_tilde = example.X_tilde, example.Y_tilde
U_tilde, V_tilde = example.U_tilde, example.V_tilde
```

Calling `make_example_views()` creates new data frames each time, so a change made inside one notebook cannot silently alter a later lesson. The shared transformation parameters are also available as `example.numeric_parameters`, `example.region_to_code`, and `example.region_mapping`.

## Check your understanding

1. Why is $(M^X)_{x3,\,age}=1$ even though the corresponding value in $\widetilde X$ is numerical?
2. What could go wrong if `region` were encoded separately in $X$ and $Y$?
3. If both original tables were complete, which objects would still be useful?
4. Why is $\widetilde X$ not described as the true completed version of $X$?
5. If a future metric uses $U$ or $V$, why must that holdout remain outside the fitting of $T_{X\cup Y}$?

## Takeaway

We now have a precise vocabulary for the entire series:

- $X,Y$: the original same-schema tables;
- $U,V$: the real utility and missingness holdouts;
- $M^X,M^Y,M^U,M^V$: binary maps of their missing cells;
- $T_{X\cup Y}$: one transformation fitted symmetrically to the compared pair; and
- $\widetilde X,\widetilde Y,\widetilde U,\widetilde V$: aligned, complete representations used only when a metric needs them.

The next lessons will introduce the comparison metrics one at a time, beginning with their mathematical definitions and then implementing them on small visible examples.